# 02 — Data Cleaning

## Objective of Data Cleaning

The objective of this notebook is to identify and handle data quality issues in the solar energy dataset before performing data integration and machine learning.

The cleaning process will focus on:

1. **Handling missing values:** Identify missing measurements in inverter and meteorological data and decide how to handle them appropriately.
2. **Correcting data types:** Ensure that timestamps, numerical measurements, and identifiers have suitable data types.
3. **Investigating invalid values:** Examine negative irradiance values, unusual wind speed readings, and other potentially invalid measurements.
4. **Checking duplicate records:** Identify exact duplicate records without removing valid readings from different inverters at the same timestamp.
5. **Validating data quality:** Verify the results of cleaning and ensure that useful observations are not removed unnecessarily.
6. **Preparing cleaned data:** Save the cleaned datasets separately for subsequent data integration, exploratory data analysis, and machine learning.

### Scope

Initially, the cleaning process will be developed and validated using the inverter and meteorological data for plant `PS_001`, along with the plant metadata. After validating the cleaning rules, they can be applied to the remaining plants.

### Data Protection

The original files in the `data/raw/` directory will remain unchanged. Cleaned datasets will be saved separately in the `data/processed/` directory to maintain reproducibility and data integrity.

In [1]:
import pandas as pd

In [2]:
metadata = pd.read_csv("../data/raw/BR-PVGen_metadata.csv")

inverter = pd.read_csv("../data/raw/BR-PVGen_inverter/PS_001.csv")

weather = pd.read_csv("../data/raw/BR-PVGen_meteorological/PS_001.csv")

In [3]:
metadata.shape

(51, 10)

In [4]:
inverter.shape

(1048575, 15)

In [5]:
weather.shape

(42336, 31)

In [6]:
metadata.head()

,brazil_federative_unit,id,is_panel_bifacial,nominal_power_mw,number_of_panels,panel_area_mm2,panel_bifaciality_coefficient,panel_efficiency_percentage,panel_temperature_coefficient,structure_type
0,SP,PS_001,True,5.0,12062,31516.36557,0.7,20.7,0.34,TRACKER
1,RJ,PS_002,True,1.0,2560,6688.93184,0.7,20.7,0.34,FIXED
2,RJ,PS_003,True,1.0,2560,6688.93184,0.7,20.7,0.34,FIXED
3,SP,PS_004,True,2.0,4480,13916.45696,0.7,21.1,0.34,FIXED
4,GO,PS_005,True,2.0,5032,13147.93165,0.7,20.7,0.34,TRACKER


In [7]:
inverter.head()

,datetime,document_count_internal_temperature_celsius,document_count_total_active_power_w,document_count_total_dc_power_w,document_count_total_reactive_power_var,internal_temperature_celsius,interpolated_keys_internal_temperature_celsius,interpolated_keys_total_active_power_w,interpolated_keys_total_dc_power_w,interpolated_keys_total_reactive_power_var,inverter_id,ps_id,total_active_power_w,total_dc_power_w,total_reactive_power_var
0,2024-03-26T00:00:00Z,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1,PS_001,NaN,NaN,NaN
1,2024-03-26T00:00:00Z,0,0,0,0,NaN,NaN,NaN,NaN,NaN,2,PS_001,NaN,NaN,NaN
2,2024-03-26T00:00:00Z,0,0,0,0,NaN,NaN,NaN,NaN,NaN,3,PS_001,NaN,NaN,NaN
3,2024-03-26T00:00:00Z,0,0,0,0,NaN,NaN,NaN,NaN,NaN,4,PS_001,NaN,NaN,NaN
4,2024-03-26T00:00:00Z,0,0,0,0,NaN,NaN,NaN,NaN,NaN,5,PS_001,NaN,NaN,NaN


In [8]:
weather.head()

,ambient_temperature_celsius,battery_voltage,datetime,document_count_ambient_temperature_celsius,document_count_battery_voltage,document_count_ghi_irradiance_wm2,document_count_gri_irradiance_wm2,document_count_panel_temperature_celsius,document_count_poa_irradiance_wm2,document_count_precipitation_accumulated_mm,...,interpolated_keys_tracker_albedo_index,interpolated_keys_wind_direction_degrees,interpolated_keys_wind_speed_ms,panel_temperature_celsius,poa_irradiance_wm2,precipitation_accumulated_mm,ps_id,tracker_albedo_index,wind_direction_degrees,wind_speed_ms
0,NaN,NaN,2024-03-26T00:00:00Z,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,PS_001,NaN,NaN,NaN
1,NaN,NaN,2024-03-26T00:15:00Z,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,PS_001,NaN,NaN,NaN
2,NaN,NaN,2024-03-26T00:30:00Z,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,PS_001,NaN,NaN,NaN
3,NaN,NaN,2024-03-26T00:45:00Z,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,PS_001,NaN,NaN,NaN
4,NaN,NaN,2024-03-26T01:00:00Z,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,PS_001,NaN,NaN,NaN


In [9]:
metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   brazil_federative_unit         51 non-null     object 
 1   id                             51 non-null     object 
 2   is_panel_bifacial              51 non-null     bool   
 3   nominal_power_mw               51 non-null     float64
 4   number_of_panels               51 non-null     int64  
 5   panel_area_mm2                 51 non-null     float64
 6   panel_bifaciality_coefficient  51 non-null     float64
 7   panel_efficiency_percentage    51 non-null     float64
 8   panel_temperature_coefficient  51 non-null     float64
 9   structure_type                 51 non-null     object 
dtypes: bool(1), float64(5), int64(1), object(3)
memory usage: 3.8+ KB


In [10]:
metadata.dtypes

brazil_federative_unit            object
id                                object
is_panel_bifacial                   bool
nominal_power_mw                 float64
number_of_panels                   int64
panel_area_mm2                   float64
panel_bifaciality_coefficient    float64
panel_efficiency_percentage      float64
panel_temperature_coefficient    float64
structure_type                    object
dtype: object

In [11]:
inverter.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 15 columns):
 #   Column                                          Non-Null Count    Dtype  
---  ------                                          --------------    -----  
 0   datetime                                        1048575 non-null  object 
 1   document_count_internal_temperature_celsius     1048575 non-null  int64  
 2   document_count_total_active_power_w             1048575 non-null  int64  
 3   document_count_total_dc_power_w                 1048575 non-null  int64  
 4   document_count_total_reactive_power_var         1048575 non-null  int64  
 5   internal_temperature_celsius                    909275 non-null   float64
 6   interpolated_keys_internal_temperature_celsius  24435 non-null    object 
 7   interpolated_keys_total_active_power_w          24435 non-null    object 
 8   interpolated_keys_total_dc_power_w              24435 non-null    object 
 9   interpolated_

In [12]:
inverter.dtypes

datetime                                           object
document_count_internal_temperature_celsius         int64
document_count_total_active_power_w                 int64
document_count_total_dc_power_w                     int64
document_count_total_reactive_power_var             int64
internal_temperature_celsius                      float64
interpolated_keys_internal_temperature_celsius     object
interpolated_keys_total_active_power_w             object
interpolated_keys_total_dc_power_w                 object
interpolated_keys_total_reactive_power_var         object
inverter_id                                         int64
ps_id                                              object
total_active_power_w                              float64
total_dc_power_w                                  float64
total_reactive_power_var                          float64
dtype: object

In [13]:
inverter.isnull().sum()

datetime                                                0
document_count_internal_temperature_celsius             0
document_count_total_active_power_w                     0
document_count_total_dc_power_w                         0
document_count_total_reactive_power_var                 0
internal_temperature_celsius                       139300
interpolated_keys_internal_temperature_celsius    1024140
interpolated_keys_total_active_power_w            1024140
interpolated_keys_total_dc_power_w                1024140
interpolated_keys_total_reactive_power_var        1024140
inverter_id                                             0
ps_id                                                   0
total_active_power_w                               139300
total_dc_power_w                                   139300
total_reactive_power_var                           139300
dtype: int64

In [14]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42336 entries, 0 to 42335
Data columns (total 31 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   ambient_temperature_celsius                    41004 non-null  float64
 1   battery_voltage                                41001 non-null  float64
 2   datetime                                       42336 non-null  object 
 3   document_count_ambient_temperature_celsius     42336 non-null  int64  
 4   document_count_battery_voltage                 42336 non-null  int64  
 5   document_count_ghi_irradiance_wm2              42336 non-null  int64  
 6   document_count_gri_irradiance_wm2              42336 non-null  int64  
 7   document_count_panel_temperature_celsius       42336 non-null  int64  
 8   document_count_poa_irradiance_wm2              42336 non-null  int64  
 9   document_count_precipitation_accumulated_mm    423

In [15]:
weather.dtypes

ambient_temperature_celsius                      float64
battery_voltage                                  float64
datetime                                          object
document_count_ambient_temperature_celsius         int64
document_count_battery_voltage                     int64
document_count_ghi_irradiance_wm2                  int64
document_count_gri_irradiance_wm2                  int64
document_count_panel_temperature_celsius           int64
document_count_poa_irradiance_wm2                  int64
document_count_precipitation_accumulated_mm        int64
document_count_tracker_albedo_index                int64
document_count_wind_direction_degrees              int64
document_count_wind_speed_ms                       int64
ghi_irradiance_wm2                               float64
gri_irradiance_wm2                               float64
interpolated_keys_ambient_temperature_celsius     object
interpolated_keys_battery_voltage                 object
interpolated_keys_ghi_irradianc

In [16]:
#Convert the inverter timestamp
inverter["datetime"] = pd.to_datetime(inverter["datetime"] , utc = True)

In [17]:
inverter["datetime"].dtypes

datetime64[ns, UTC]

In [18]:
weather.isnull().sum()

ambient_temperature_celsius                       1332
battery_voltage                                   1335
datetime                                             0
document_count_ambient_temperature_celsius           0
document_count_battery_voltage                       0
document_count_ghi_irradiance_wm2                    0
document_count_gri_irradiance_wm2                    0
document_count_panel_temperature_celsius             0
document_count_poa_irradiance_wm2                    0
document_count_precipitation_accumulated_mm          0
document_count_tracker_albedo_index                  0
document_count_wind_direction_degrees                0
document_count_wind_speed_ms                         0
ghi_irradiance_wm2                                1317
gri_irradiance_wm2                                1324
interpolated_keys_ambient_temperature_celsius    42279
interpolated_keys_battery_voltage                42280
interpolated_keys_ghi_irradiance_wm2             42278
interpolat

In [19]:
weather["datetime"] = pd.to_datetime(weather["datetime"] , utc = True)

In [20]:
weather["datetime"].dtype

datetime64[ns, UTC]

## Handle Missing Values

In [21]:
inverter.isnull().mean().mul(100).round(2)

datetime                                           0.00
document_count_internal_temperature_celsius        0.00
document_count_total_active_power_w                0.00
document_count_total_dc_power_w                    0.00
document_count_total_reactive_power_var            0.00
internal_temperature_celsius                      13.28
interpolated_keys_internal_temperature_celsius    97.67
interpolated_keys_total_active_power_w            97.67
interpolated_keys_total_dc_power_w                97.67
interpolated_keys_total_reactive_power_var        97.67
inverter_id                                        0.00
ps_id                                              0.00
total_active_power_w                              13.28
total_dc_power_w                                  13.28
total_reactive_power_var                          13.28
dtype: float64

In [22]:
#Investigate missing active power
inverter.loc[inverter["total_active_power_w"].isnull() , "document_count_total_active_power_w"].value_counts(dropna=False)

document_count_total_active_power_w
0    139300
Name: count, dtype: int64

In [23]:
inverter.loc[
    inverter["total_active_power_w"].isnull(),
    [
        "datetime",
        "inverter_id",
        "document_count_total_active_power_w",
        "total_active_power_w"
    ]
].head(10)

,datetime,inverter_id,document_count_total_active_power_w,total_active_power_w
0,2024-03-26 00:00:00+00:00,1,0,NaN
1,2024-03-26 00:00:00+00:00,2,0,NaN
2,2024-03-26 00:00:00+00:00,3,0,NaN
3,2024-03-26 00:00:00+00:00,4,0,NaN
4,2024-03-26 00:00:00+00:00,5,0,NaN
5,2024-03-26 00:00:00+00:00,6,0,NaN
6,2024-03-26 00:00:00+00:00,7,0,NaN
7,2024-03-26 00:00:00+00:00,8,0,NaN
8,2024-03-26 00:00:00+00:00,9,0,NaN
9,2024-03-26 00:00:00+00:00,10,0,NaN


In [24]:
weather.isnull().mean().mul(100).round(2)

ambient_temperature_celsius                       3.15
battery_voltage                                   3.15
datetime                                          0.00
document_count_ambient_temperature_celsius        0.00
document_count_battery_voltage                    0.00
document_count_ghi_irradiance_wm2                 0.00
document_count_gri_irradiance_wm2                 0.00
document_count_panel_temperature_celsius          0.00
document_count_poa_irradiance_wm2                 0.00
document_count_precipitation_accumulated_mm       0.00
document_count_tracker_albedo_index               0.00
document_count_wind_direction_degrees             0.00
document_count_wind_speed_ms                      0.00
ghi_irradiance_wm2                                3.11
gri_irradiance_wm2                                3.13
interpolated_keys_ambient_temperature_celsius    99.87
interpolated_keys_battery_voltage                99.87
interpolated_keys_ghi_irradiance_wm2             99.86
interpolat

In [25]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    "document_count_ghi_irradiance_wm2"
].value_counts(dropna=False)

document_count_ghi_irradiance_wm2
0    1317
Name: count, dtype: int64

In [26]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    ["datetime", "ghi_irradiance_wm2"]
].head(10)

,datetime,ghi_irradiance_wm2
0,2024-03-26 00:00:00+00:00,NaN
1,2024-03-26 00:15:00+00:00,NaN
2,2024-03-26 00:30:00+00:00,NaN
3,2024-03-26 00:45:00+00:00,NaN
4,2024-03-26 01:00:00+00:00,NaN
5,2024-03-26 01:15:00+00:00,NaN
6,2024-03-26 01:30:00+00:00,NaN
7,2024-03-26 01:45:00+00:00,NaN
8,2024-03-26 02:00:00+00:00,NaN
9,2024-03-26 02:15:00+00:00,NaN


In [27]:
weather[
    [
        "ghi_irradiance_wm2",
        "poa_irradiance_wm2",
        "ambient_temperature_celsius",
        "panel_temperature_celsius",
        "wind_speed_ms",
        "wind_direction_degrees",
        "precipitation_accumulated_mm"
    ]
].isnull().sum()


ghi_irradiance_wm2              1317
poa_irradiance_wm2              1322
ambient_temperature_celsius     1332
panel_temperature_celsius       1336
wind_speed_ms                   1341
wind_direction_degrees          1338
precipitation_accumulated_mm      71
dtype: int64

In [28]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    [
        "datetime",
        "ghi_irradiance_wm2",
        "poa_irradiance_wm2",
        "ambient_temperature_celsius",
        "panel_temperature_celsius",
        "wind_speed_ms"
    ]
].head(10)

,datetime,ghi_irradiance_wm2,poa_irradiance_wm2,ambient_temperature_celsius,panel_temperature_celsius,wind_speed_ms
0,2024-03-26 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN
1,2024-03-26 00:15:00+00:00,NaN,NaN,NaN,NaN,NaN
2,2024-03-26 00:30:00+00:00,NaN,NaN,NaN,NaN,NaN
3,2024-03-26 00:45:00+00:00,NaN,NaN,NaN,NaN,NaN
4,2024-03-26 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN
5,2024-03-26 01:15:00+00:00,NaN,NaN,NaN,NaN,NaN
6,2024-03-26 01:30:00+00:00,NaN,NaN,NaN,NaN,NaN
7,2024-03-26 01:45:00+00:00,NaN,NaN,NaN,NaN,NaN
8,2024-03-26 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN
9,2024-03-26 02:15:00+00:00,NaN,NaN,NaN,NaN,NaN


In [29]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    ["datetime", "document_count_ghi_irradiance_wm2"]
].head(30)

,datetime,document_count_ghi_irradiance_wm2
0,2024-03-26 00:00:00+00:00,0
1,2024-03-26 00:15:00+00:00,0
2,2024-03-26 00:30:00+00:00,0
3,2024-03-26 00:45:00+00:00,0
4,2024-03-26 01:00:00+00:00,0
5,2024-03-26 01:15:00+00:00,0
6,2024-03-26 01:30:00+00:00,0
7,2024-03-26 01:45:00+00:00,0
8,2024-03-26 02:00:00+00:00,0
9,2024-03-26 02:15:00+00:00,0


In [30]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    "datetime"
].min(), weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    "datetime"
].max()

(Timestamp('2024-03-26 00:00:00+0000', tz='UTC'),
 Timestamp('2025-06-09 23:45:00+0000', tz='UTC'))

find the longest continuous periods when GHI (sunlight) measurements are missing in your solar dataset.

In [31]:
#Identify missing GHI value
missing_ghi = weather["ghi_irradiance_wm2"].isnull()

In [32]:
#Assign a group number to consecutive patterns
groups = missing_ghi.ne(missing_ghi.shift()).cumsum()

In [33]:
missing_ghi_gaps = (
    weather.loc[missing_ghi]
    .groupby(groups[missing_ghi])
    .agg(
        start_time=("datetime", "min"),
        end_time=("datetime", "max"),
        missing_readings=("datetime", "size")
    )
)

In [34]:
missing_ghi_gaps["duration_minutes"] = (
    missing_ghi_gaps["missing_readings"] * 15
)

In [35]:
missing_ghi_gaps.sort_values(
    "missing_readings", ascending=False
).head(10)

,start_time,end_time,missing_readings,duration_minutes
ghi_irradiance_wm2,,,,
37,2024-09-07 03:00:00+00:00,2024-09-09 11:00:00+00:00,225,3375
3,2024-03-30 22:45:00+00:00,2024-04-01 18:45:00+00:00,177,2655
61,2024-12-26 08:15:00+00:00,2024-12-27 22:15:00+00:00,153,2295
57,2024-12-09 22:30:00+00:00,2024-12-10 19:15:00+00:00,84,1260
5,2024-04-02 21:00:00+00:00,2024-04-03 14:45:00+00:00,72,1080
1,2024-03-26 00:00:00+00:00,2024-03-26 17:00:00+00:00,69,1035
27,2024-06-11 20:45:00+00:00,2024-06-12 11:15:00+00:00,59,885
31,2024-08-12 23:45:00+00:00,2024-08-13 11:00:00+00:00,46,690
75,2025-02-13 08:15:00+00:00,2025-02-13 19:15:00+00:00,45,675


Insights from GHI Missing Values (Simple)
1. Long gaps: Some sunlight readings are missing continuously for many hours.
2. Longest gap: 225 readings are missing from 7 September to 9 September 2024, covering approximately 56 hours and 15 minutes.
3. Different months: Missing readings occur across several months, not just one period.
4. Possible issue: The data may have been unavailable for an extended time, but we don't yet know the exact reason.
5. Cleaning decision: We should not replace all missing values with zero or fill long gaps using interpolation, because that could create incorrect data.
Conclusion: The GHI column contains long periods of missing data, so we need to investigate other weather columns before deciding how to handle these gaps.

### Check Missing Values Across Weather Measurements

In this step, I will check whether other weather measurements are missing at the same timestamps as GHI.

This analysis will help me understand whether missing values occur independently or affect multiple weather measurements together. The findings will guide the missing-value treatment strategy.

In [36]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    [
        "datetime",
        "ghi_irradiance_wm2",
        "poa_irradiance_wm2",
        "ambient_temperature_celsius",
        "panel_temperature_celsius",
        "wind_speed_ms",
        "wind_direction_degrees"
    ]
].isnull().sum()

datetime                          0
ghi_irradiance_wm2             1317
poa_irradiance_wm2             1316
ambient_temperature_celsius    1317
panel_temperature_celsius      1317
wind_speed_ms                  1317
wind_direction_degrees         1317
dtype: int64

Conclusion: Missing weather values mostly occur together at the same timestamps. This suggests that multiple weather measurements may have been unavailable during the same periods.

In [37]:
weather.loc[
    weather["ghi_irradiance_wm2"].isnull(),
    [
        "document_count_ghi_irradiance_wm2",
        "document_count_poa_irradiance_wm2",
        "document_count_ambient_temperature_celsius",
        "document_count_panel_temperature_celsius",
        "document_count_wind_speed_ms"
    ]
].value_counts(dropna=False)

document_count_ghi_irradiance_wm2  document_count_poa_irradiance_wm2  document_count_ambient_temperature_celsius  document_count_panel_temperature_celsius  document_count_wind_speed_ms
0                                  0                                  0                                           0                                         0                               1316
                                   1                                  0                                           0                                         0                                  1
Name: count, dtype: int64

## Simpler Handle missing value

### Select Important Columns

In [38]:
inverter_clean = inverter[[
    "datetime", "inverter_id" , "ps_id" , "total_active_power_w"
]].copy()

In [39]:
weather_clean = weather[[
    "datetime" , "ps_id" ,
    "ghi_irradiance_wm2" , 
    "poa_irradiance_wm2", 
    "ambient_temperature_celsius",
    "panel_temperature_celsius",
    "wind_speed_ms"
]].copy()

In [40]:
inverter_clean.head()

,datetime,inverter_id,ps_id,total_active_power_w
0,2024-03-26 00:00:00+00:00,1,PS_001,NaN
1,2024-03-26 00:00:00+00:00,2,PS_001,NaN
2,2024-03-26 00:00:00+00:00,3,PS_001,NaN
3,2024-03-26 00:00:00+00:00,4,PS_001,NaN
4,2024-03-26 00:00:00+00:00,5,PS_001,NaN


In [41]:
weather_clean.head()

,datetime,ps_id,ghi_irradiance_wm2,poa_irradiance_wm2,ambient_temperature_celsius,panel_temperature_celsius,wind_speed_ms
0,2024-03-26 00:00:00+00:00,PS_001,NaN,NaN,NaN,NaN,NaN
1,2024-03-26 00:15:00+00:00,PS_001,NaN,NaN,NaN,NaN,NaN
2,2024-03-26 00:30:00+00:00,PS_001,NaN,NaN,NaN,NaN,NaN
3,2024-03-26 00:45:00+00:00,PS_001,NaN,NaN,NaN,NaN,NaN
4,2024-03-26 01:00:00+00:00,PS_001,NaN,NaN,NaN,NaN,NaN


In [42]:
inverter_clean.shape

(1048575, 4)

In [43]:
weather_clean.shape

(42336, 7)

In [65]:
inverter_clean.isnull().sum()

datetime                     0
inverter_id                  0
ps_id                        0
total_active_power_w    139300
dtype: int64

In [64]:
weather_clean.isnull().sum()

datetime                       0
ps_id                          0
ghi_irradiance_wm2             0
poa_irradiance_wm2             0
ambient_temperature_celsius    0
panel_temperature_celsius      0
wind_speed_ms                  0
dtype: int64

In [46]:
#Remove rows without active power
#inverter_clean = inverter_clean.dropna(subset=["total_active_power_w"]).copy()

In [47]:
#inverter_clean.shape

In [48]:
inverter_clean["total_active_power_w"].isnull().sum()

np.int64(139300)

In [49]:
inverter_clean['total_active_power_w'].isnull().mean()*100

np.float64(13.284695896812341)

In [50]:
#Check recorded zero values
(inverter_clean['total_active_power_w']==0).sum()

np.int64(454752)

In [51]:
#Check recorded zero values
(inverter_clean['total_active_power_w']==0).mean()*100

np.float64(43.36857163293041)

In [52]:
#Check negative irradiance values
(weather_clean['ghi_irradiance_wm2'] < 0 ).sum()

np.int64(17595)

In [53]:
#Check negative irradiance values
(weather_clean["poa_irradiance_wm2"] < 0 ).sum()

np.int64(19216)

In [54]:
#Replace negative irradiance with zero
weather_clean.loc[weather_clean['ghi_irradiance_wm2'] < 0 , "ghi_irradiance_wm2" ]= 0

weather_clean.loc[ weather_clean["poa_irradiance_wm2"] < 0,"poa_irradiance_wm2"] = 0


We want to prevent negative irradiance values from being treated as valid incoming sunlight in our analysis.

In [55]:
weather_clean[
    ["ghi_irradiance_wm2", "poa_irradiance_wm2"]
].min()

ghi_irradiance_wm2    0.0
poa_irradiance_wm2    0.0
dtype: float64

In [63]:
#Check missing values again
weather_clean.isnull().sum()

datetime                       0
ps_id                          0
ghi_irradiance_wm2             0
poa_irradiance_wm2             0
ambient_temperature_celsius    0
panel_temperature_celsius      0
wind_speed_ms                  0
dtype: int64

In [57]:
inverter_clean.isnull().sum()


datetime                     0
inverter_id                  0
ps_id                        0
total_active_power_w    139300
dtype: int64

In [60]:
weather_features = [
    "ghi_irradiance_wm2",
    "poa_irradiance_wm2",
    "ambient_temperature_celsius",
    "panel_temperature_celsius",
    "wind_speed_ms"
]

for column in weather_features:
    weather_clean[column] = weather_clean[column].fillna(
        weather_clean[column].median()
    )

In [62]:
weather_clean[weather_features].isnull().sum()

ghi_irradiance_wm2             0
poa_irradiance_wm2             0
ambient_temperature_celsius    0
panel_temperature_celsius      0
wind_speed_ms                  0
dtype: int64

In [61]:
#Save the processed datasets
inverter_clean.to_csv(
    "../data/processed/PS_001_inverter_clean.csv",
    index=False
)

weather_clean.to_csv(
    "../data/processed/PS_001_weather_clean.csv",
    index=False
) 

In my Solar Energy Generation Forecasting project, I found missing values in both inverter and weather data. The inverter active-power column contained 139,300 missing values, approximately 13.28% of the records.

For weather data, I used median imputation for five numerical features: GHI irradiance, POA irradiance, ambient temperature, panel temperature, and wind speed. I then verified that these columns had no missing values.

For inverter active power, I retained missing values because missing measurements should not automatically be treated as zero generation. I plan to exclude rows with missing target values during model preparation. I also replaced negative GHI and POA readings with zero as a separate physical-validity cleaning step.

This approach allowed me to preserve the inverter records while preparing the weather data for integration and forecasting.